**SQL Problem Statement**

You are given two tables: `agents` and `deliveries`.

### Table: `agents`

| id | name  | city   |
| -- | ----- | ------ |
| 1  | Aarav | Mumbai |
| 2  | Priya | Mumbai |
| 3  | Rahul | Delhi  |
| 4  | Sneha | Delhi  |
| 5  | Amit  | Mumbai |
| 6  | Neha  | Delhi  |

### Table: `deliveries`

| d_id | agent_id | del_date   | status  |
| ---- | -------- | ---------- | ------- |
| 1    | 1        | 2024-03-01 | on_time |
| 2    | 1        | 2024-03-02 | on_time |
| 3    | 1        | 2024-03-03 | late    |
| 4    | 1        | 2024-03-04 | on_time |
| 5    | 2        | 2024-03-01 | on_time |
| 6    | 2        | 2024-03-02 | on_time |
| 7    | 2        | 2024-03-03 | on_time |
| 8    | 3        | 2024-03-01 | on_time |
| 9    | 3        | 2024-03-02 | late    |
| 10   | 3        | 2024-03-03 | late    |
| 11   | 4        | 2024-03-01 | on_time |
| 12   | 4        | 2024-03-02 | on_time |
| 13   | 4        | 2024-03-03 | on_time |
| 14   | 4        | 2024-03-04 | on_time |
| 15   | 5        | 2024-03-01 | late    |
| 16   | 5        | 2024-03-02 | on_time |
| 17   | 6        | 2024-03-01 | on_time |
| 18   | 6        | 2024-03-02 | late    |
| 19   | 6        | 2024-03-03 | on_time |

### Task

For each delivery agent:

1. Calculate the total number of deliveries (`total`).

2. Calculate the number of on-time deliveries (`on_time`).

3. Calculate the on-time delivery percentage (`ot_pct`) as:

   ```
   (on_time deliveries / total deliveries) * 100
   ```

4. Rank agents within their respective city based on:

   * Highest on-time percentage (`ot_pct`) first.
   * If two agents have the same percentage, rank the agent with more total deliveries higher.
   * If still tied, rank alphabetically by agent name.

Return the following columns:

| name | city | total | on_time | ot_pct | city_rank |

### Expected Output

| name  | city   | total | on_time | ot_pct | city_rank |
| ----- | ------ | ----- | ------- | ------ | --------- |
| Priya | Mumbai | 3     | 3       | 100.0  | 1         |
| Aarav | Mumbai | 4     | 3       | 75.0   | 2         |
| Amit  | Mumbai | 2     | 1       | 50.0   | 3         |
| Sneha | Delhi  | 4     | 4       | 100.0  | 1         |
| Neha  | Delhi  | 3     | 2       | 66.7   | 2         |
| Rahul | Delhi  | 3     | 1       | 33.3   | 3         |

Write an SQL query to generate the above result.


In [0]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as f
from pyspark.sql import Window 
spark = SparkSession.builder.appName("AgentsDeliveries").getOrCreate()
# Agents DataFrame
agents_data = [
   (1, "Aarav", "Mumbai"),
   (2, "Priya", "Mumbai"),
   (3, "Rahul", "Delhi"),
   (4, "Sneha", "Delhi"),
   (5, "Amit", "Mumbai"),
   (6, "Neha", "Delhi")
]
agents_columns = ["id", "name", "city"]
agents_df = spark.createDataFrame(agents_data, agents_columns)
# Deliveries DataFrame
deliveries_data = [
   (1, 1, "2024-03-01", "on_time"),
   (2, 1, "2024-03-02", "on_time"),
   (3, 1, "2024-03-03", "late"),
   (4, 1, "2024-03-04", "on_time"),
   (5, 2, "2024-03-01", "on_time"),
   (6, 2, "2024-03-02", "on_time"),
   (7, 2, "2024-03-03", "on_time"),
   (8, 3, "2024-03-01", "on_time"),
   (9, 3, "2024-03-02", "late"),
   (10, 3, "2024-03-03", "late"),
   (11, 4, "2024-03-01", "on_time"),
   (12, 4, "2024-03-02", "on_time"),
   (13, 4, "2024-03-03", "on_time"),
   (14, 4, "2024-03-04", "on_time"),
   (15, 5, "2024-03-01", "late"),
   (16, 5, "2024-03-02", "on_time"),
   (17, 6, "2024-03-01", "on_time"),
   (18, 6, "2024-03-02", "late"),
   (19, 6, "2024-03-03", "on_time")
]
deliveries_columns = ["d_id", "agent_id", "del_date", "status"]
deliveries_df = spark.createDataFrame(deliveries_data, deliveries_columns)
# Optional: convert date column to DateType
from pyspark.sql.functions import col
from pyspark.sql.types import DateType
deliveries_df = deliveries_df.withColumn(
   "del_date",
   col("del_date").cast(DateType())
)
# Show DataFrames
agents_df.show()
deliveries_df.show()

In [0]:
result_df = (
    agents_df.join(deliveries_df, agents_df.id == deliveries_df.agent_id)
    .groupBy("name", "city")
    .agg(
        f.count(deliveries_df.agent_id).alias("total"),
        f.sum(f.when(f.col("status") == "on_time", 1).otherwise(0)).alias("on_time"),
    )
    .select(
        f.col("name"),
        f.col("city"),
        f.col("total"),
        f.col("on_time"),
        f.round((f.col("on_time") * 100 / f.col("total")), 2).alias("ot_pct"),
    )
    .withColumn(
        "city_rank",
        f.dense_rank().over(Window.partitionBy("city").orderBy(f.desc("ot_pct"))),
    )
)
display(result_df)